# Ardoise à tresse

Une **tresse** à n brins s'écrit comme un mot en σ₁ … σ_(n-1) : σ_i croise le brin i et le brin i+1, σ_i⁻¹ les croise dans
l'autre sens. Ce cahier montre deux manières de la représenter :

- le **diagramme de tresse** classique : les brins se croisent, de haut en bas ;
- l'**action sur le groupe libre** : on place n trous dans un disque ; σ_i fait faire un demi-tour aux trous i et i+1 l'un
  autour de l'autre, et entraîne les lacets x₁ … x_n (x_i part d'un point-base, le « clou », fait le tour du trou i et
  revient). L'image du lacet x_i par la tresse est un nouveau lacet, que l'on lit comme un mot en x₁ … x_n.

Les cellules qui suivent écrivent ce calcul (l'action d'Artin) et le dessin du diagramme ; la cellule « L'ardoise »
est l'**ardoise** : on y écrit un mot de tresse, ou l'on en choisit un parmi douze, et l'on voit ses deux représentations et
les lacets dessinés au fil de la tresse, en film continu, de quatre manières. Un dernier chapitre, facultatif, montre
que les mêmes lacets savent dessiner autre chose qu'une tresse.

Une tresse est une liste d'entiers non nuls : `k` pour σ_k, `-k` pour σ_k⁻¹, lue dans l'ordre (la première lettre joue la
première). Les signes et le sens des demi-tours sont expliqués dans `conventions_de_signe.md`, à la racine du dépôt.

Ce cahier se lance depuis le répertoire `ipynbks/` du dépôt (c'est ce que fait JupyterLab en l'ouvrant) : il utilise les
programmes de `python/`, pour les films (`Conforme.py`, `Arcs.py`, `Triangulation.py`) et pour l'interface
(`Interface.py` : lecture du mot écrit, lecteur de film, boutons). Exécuter les cellules jusqu'à l'ardoise.

In [ ]:
import sys
sys.path.insert(0, '../python')

import cairo
from IPython.display import Image

from Interface import Ardoise, COULEURS, png

## Le groupe libre et l'action d'Artin

Un élément du groupe libre F_n est un mot en x₁ … x_n et leurs inverses : on le note comme une liste d'entiers non nuls,
`i` pour x_i et `-i` pour x_i⁻¹. Deux lettres inverses l'une de l'autre qui se suivent s'annulent : `simplifie` les
supprime.

La tresse σ_i agit sur F_n (convention de Birman) en laissant les autres générateurs fixes et en envoyant

- pour σ_i : x_i ↦ x_i x_(i+1) x_i⁻¹ et x_(i+1) ↦ x_i ;
- pour σ_i⁻¹ : x_i ↦ x_(i+1) et x_(i+1) ↦ x_(i+1)⁻¹ x_i x_(i+1).

L'automorphisme d'une tresse s'obtient en composant ceux de ses lettres. Comme on lit la tresse dans l'ordre, la
première lettre agissant la première, on parcourt le mot **à l'envers** : l'automorphisme de `[a₁, …, aₙ]` est
φ_aₙ ∘ … ∘ φ_a₁.

In [ ]:
def morphisme_identite(n):
    """L'identité de F_n : l'image de chaque x_i est x_i. `images[i]` est l'image de x_(i+1)."""
    return [[e] for e in range(1, n + 1)]

def inverse(mot):
    return [-e for e in reversed(mot)]

def conjugaison_locale(sigma, images):
    """Compose l'automorphisme `images` avec l'action de la lettre σ (modifie `images`)."""
    i = abs(sigma)
    x1, x2 = images[i - 1], images[i]
    if sigma > 0:
        images[i - 1] = x1 + x2 + inverse(x1)       # x_i ↦ x_i x_(i+1) x_i⁻¹
        images[i] = x1                              # x_(i+1) ↦ x_i
    else:
        images[i - 1] = x2                          # x_i ↦ x_(i+1)
        images[i] = inverse(x2) + x1 + x2           # x_(i+1) ↦ x_(i+1)⁻¹ x_i x_(i+1)

def simplifie(mot):
    """Supprime les paires de lettres inverses qui se suivent (modifie `mot`)."""
    i = 0
    while i < len(mot) - 1:
        if mot[i] == -mot[i + 1]:
            del mot[i:i + 2]
            i = max(i - 1, 0)
        else:
            i += 1

def calcule_autofn_de_tresse(tresse, nb_trous=None):
    """L'automorphisme de F_n de la tresse : la liste des images de x₁ … x_n, avec n = nombre de trous (par défaut
    celui de la tresse : sa plus grande lettre + 1)."""
    if nb_trous is None: nb_trous = max(map(abs, tresse), default=5) + 1
    images = morphisme_identite(nb_trous)
    for sigma in reversed(tresse):
        conjugaison_locale(sigma, images)
    for mot in images:
        simplifie(mot)
    return images

Le résultat est la liste des images de x₁, x₂, … dans cet ordre. Pour σ₁ seule, `[[1, 2, -1], [1]]` se lit
x₁ ↦ x₁x₂x₁⁻¹ et x₂ ↦ x₁. Pour σ₁σ₂⁻¹ (σ₁ d'abord, puis σ₂⁻¹), on attend
x₁ ↦ x₁x₃x₁⁻¹, x₂ ↦ x₁ et x₃ ↦ x₃⁻¹x₂x₃ : σ₁ envoie x₁ sur x₁x₂x₁⁻¹, puis σ₂⁻¹ remplace x₂ par x₃ ; σ₁ envoie x₂ sur x₁,
que σ₂⁻¹ ne touche pas ; σ₁ ne touche pas x₃, que σ₂⁻¹ envoie sur x₃⁻¹x₂x₃.

In [ ]:
print(calcule_autofn_de_tresse([1]))
print(calcule_autofn_de_tresse([1, -2]))

## Le diagramme de tresse

Dessiné de haut en bas, la première lettre en haut. Pour σ_i positif, le brin qui va d'en haut à droite vers en bas à
gauche passe devant ; pour σ_i⁻¹, c'est l'autre. Les couleurs sont celles des lacets : le brin de la colonne i au départ
est x_i. Le brin de dessous est interrompu au croisement.

In [ ]:
def diagramme_de_tresse(tresse, maille=60, largeur_brin=6):
    """La surface Cairo du diagramme de tresse (première lettre en haut)."""
    n = max(map(abs, tresse), default=0) + 1                  # nombre de brins
    marge = maille // 2
    surface = cairo.ImageSurface(cairo.FORMAT_ARGB32, n * maille, (len(tresse) + 1) * maille)
    ctx = cairo.Context(surface)
    ctx.set_source_rgb(1, 1, 1)
    ctx.paint()
    ctx.set_line_width(largeur_brin)
    ctx.set_line_cap(cairo.LINE_CAP_ROUND)
    couleurs = [COULEURS[i % len(COULEURS)] for i in range(n)]   # couleur du brin dans chaque colonne
    x = lambda colonne: (colonne + 0.5) * maille

    def segment(couleur, c0, y0, c1, y1, de=0.0, a=1.0):
        # le segment de (colonne c0, hauteur y0) à (colonne c1, hauteur y1), de la fraction `de` à la fraction `a`
        ctx.set_source_rgb(*couleur)
        p = lambda t: (x(c0) + t * (x(c1) - x(c0)), y0 + t * (y1 - y0))
        ctx.move_to(*p(de)); ctx.line_to(*p(a)); ctx.stroke()

    for ligne, sigma in enumerate(tresse):
        y0, y1 = marge + ligne * maille, marge + (ligne + 1) * maille
        k = abs(sigma) - 1                                    # le croisement est entre les colonnes k et k + 1
        for c in range(n):
            if c not in (k, k + 1): segment(couleurs[c], c, y0, c, y1)
        trou = 0.3                                            # largeur de l'interruption (fraction du croisement)
        # le brin de gauche va vers la droite, et inversement ; σ_i positif : celui de droite passe devant
        vers_la_droite, vers_la_gauche = (couleurs[k], k, k + 1), (couleurs[k + 1], k + 1, k)
        sous, dessus = (vers_la_droite, vers_la_gauche) if sigma > 0 else (vers_la_gauche, vers_la_droite)
        couleur, depart, arrivee = sous
        segment(couleur, depart, y0, arrivee, y1, 0, 0.5 - trou / 2)
        segment(couleur, depart, y0, arrivee, y1, 0.5 + trou / 2, 1)
        segment(dessus[0], dessus[1], y0, dessus[2], y1)
        couleurs[k], couleurs[k + 1] = couleurs[k + 1], couleurs[k]
    return surface

In [ ]:
Image(data=png(diagramme_de_tresse([1, -2, 1, -2, 3])))

## Les lacets au fil de la tresse

Pendant σ_i, les trous i et i+1 tournent l'un autour de l'autre et entraînent les lacets : la forme finale est celle de
l'automorphisme calculé plus haut. Les films de l'ardoise dessinent ce mouvement de quatre manières, toutes écrites dans
`python/` :

- le **diagramme de courbes** (`Arcs.DiagrammeCourbes`) : un seul point fixe, à gauche, et pour arcs les segments de
  l'axe, du point fixe au premier trou puis d'un trou au suivant ; la tresse les emporte et ils portent la même
  information que l'action sur le groupe libre ;
- les **lacets** (`Conforme.dessins_conforme`) : des arcs de cercle perpendiculaires aux segments d'une ligne brisée qui
  se plie, par représentation conforme (Schwarz–Christoffel), sans croisement ;
- la **triangulation mobile** (`Triangulation.Film`) : les lacets sont posés sur une triangulation dont les trous
  tournent, les points des lacets suivent leur arête, et les arêtes qui cessent d'être de Delaunay sont retournées sans
  changer la forme des lacets ;
- les **rubans** (`Arcs.FilmArcs`) : sans clou, chaque lacet est le bord d'un ruban mince tendu entre un trou fixe, en bas,
  et son trou voyageur.

Chaque film s'obtient par une fonction de `Interface.py`, qui renvoie un lecteur (curseur et bouton de lecture) :

In [ ]:
from Interface import film_rubans
film_rubans([1, -2])

## L'ardoise

Écrire un mot (`1 -2 3`, `σ1 σ2^-1`, `σ₁σ₂⁻¹` ou `s1^3 s2`, puis Entrée ou « Dessiner ») ou choisir un exemple. Le diagramme
de tresse et l'action sur le groupe libre, calculés par les fonctions ci-dessus, s'affichent aussitôt ; chaque film est
calculé quand on ouvre son onglet (de quelques secondes à quelques minutes pour un mot long : le noyau est occupé pendant
le calcul).

In [ ]:
ardoise = Ardoise(diagramme_de_tresse, calcule_autofn_de_tresse)
ardoise.affiche()

## Un peu de poésie

Une tresse est un mouvement de points dans un disque : les demi-tours de σ_i n'en sont qu'un cas. Si les trous suivent des
chemins quelconques, sans jamais se rencontrer, les lacets sont emportés de la même façon, avec la même topologie, et le
calcul des films ne change pas. Il n'y a plus de mot de tresse, seulement une chorégraphie des trous.

**Un escargot** : une spirale de rubans, trous fixes à gauche ; un anneau de trous tourne pour la coquille, puis la tête et
les yeux sortent (`Escargot.py`). Un clic lance le calcul (une dizaine de secondes) : rien ne se calcule en « exécutant
toutes les cellules ».

In [ ]:
from Interface import a_la_demande, film_escargot, film_bouquet
a_la_demande(film_escargot, "Dessiner l'escargot")

**Un bouquet d'iris** : les trous dorment sous la terre, puis montent depuis le clou et tirent des lacets, les tiges. Chaque
fleur est un groupe de trous : cinq sépales qui forment une cage qui ne tourne pas, et quatre pétales qui tournent d'un tour
à l'intérieur, de sorte que leurs lacets s'enroulent en couches (le bourgeon), puis sortent entre les sépales (`Bouquet.py`).
Un clic lance le calcul (environ trois quarts de minute).

In [ ]:
a_la_demande(film_bouquet, 'Faire pousser le bouquet')